# 리간드 스크리닝 — 여러 화합물 일괄 도킹 & 랭킹

노트북 1에서 선택한 PDB 구조에 여러 리간드를 도킹하여 랭킹합니다.

## 워크플로우
1. 타겟 PDB 설정 + 리간드 SMILES 입력
2. Docking Box 설정 (auto / residue / manual) + 3D 확인
3. 전체 리간드 일괄 도킹
4. 결과 분석 (Score, LE, RMSD, ProLIF)
5. Top 포즈 3D 비교
6. 내보내기 (SDF + CSV + PyMOL)


## 0. 환경 설정


In [ ]:
#@title 환경 설치 (처음 1회) {display-mode: "form"}
import subprocess, sys, os

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

pip_install('rdkit', 'meeko', 'openbabel-wheel')
pip_install('pdbfixer', 'openmm')
pip_install('py3Dmol', 'prolif', 'MDAnalysis')
pip_install('seaborn', 'pandas', 'matplotlib', 'requests')
try: pip_install('pymol-open-source')
except: pass

# smina
BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)
smina_path = os.path.join(BIN_DIR, 'smina')
if not os.path.exists(smina_path):
    import stat, urllib.request
    print('Downloading smina...')
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    os.chmod(smina_path, os.stat(smina_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
os.environ['PATH'] = BIN_DIR + ':' + os.environ['PATH']
print('Done.')


In [ ]:
#@title 라이브러리 로드 {display-mode: "form"}
import warnings; warnings.filterwarnings('ignore')
import os, subprocess, urllib.request, time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, PandasTools, SanitizeFlags
from openbabel import pybel
from meeko import MoleculePreparation, PDBQTWriterLegacy
from pdbfixer import PDBFixer
from openmm.app import PDBFile
import MDAnalysis as mda
import prolif as plf

BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
WORK_DIR = '/content/ligand_screening' if os.path.exists('/content') else os.path.join(os.path.expanduser('~'), 'ligand_screening')
os.makedirs(WORK_DIR, exist_ok=True)

print('All libraries loaded.')


## 1. 타겟 & 리간드 입력


In [ ]:
#@title 1-1. 타겟 PDB 설정 {display-mode: "form"}
TARGET_PDB = "6nzp"  #@param {type:"string"}
TARGET_CHAIN = "A"   #@param {type:"string"}

print(f'Target: {TARGET_PDB} chain {TARGET_CHAIN}')


In [ ]:
#@title 1-2. 리간드 SMILES 입력 {display-mode: "form"}

# 직접 입력
LIGANDS = [
    {"name": "Erlotinib",     "smiles": "C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1"},
    {"name": "Gefitinib",     "smiles": "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1"},
    {"name": "Afatinib",      "smiles": "CN(C)C/C=C/C(=O)Nc1cc2c(Nc3ccc(F)c(Cl)c3)ncnc2cc1OC1CCOC1"},
    {"name": "Osimertinib",   "smiles": "C=CC(=O)Nc1cc(OC)c(Nc2nccc(-c3cn(C)c4ccccc34)n2)cc1N(C)CCN(C)C"},
    {"name": "Imatinib",      "smiles": "Cc1ccc(NC(=O)c2ccc(CN3CCN(C)CC3)cc2)cc1Nc1nccc(-c2cccnc2)n1"},
]

# 또는 CSV에서 로드:
# df_input = pd.read_csv('my_ligands.csv')  # columns: name, smiles
# LIGANDS = df_input.to_dict('records')

print(f'{len(LIGANDS)} ligands loaded:')
for lig in LIGANDS:
    print(f"  {lig['name']:20s} {lig['smiles'][:50]}...")


In [ ]:
#@title 1-3. 리간드 2D 구조 확인 {display-mode: "form"}
mols = []
legends = []
for lig in LIGANDS:
    mol = Chem.MolFromSmiles(lig['smiles'])
    if mol:
        mols.append(mol)
        legends.append(lig['name'])
    else:
        print(f"Invalid SMILES: {lig['name']}")

Draw.MolsToGridImage(mols, legends=legends, molsPerRow=5, subImgSize=(300, 220))


## 2. 타겟 구조 준비


In [ ]:
#@title 2-1. PDB 구조 준비 {display-mode: "form"}
os.chdir(WORK_DIR)
pdb_id = TARGET_PDB.lower()

# Fetch
pdb_path = os.path.join(WORK_DIR, f'{pdb_id}.pdb')
if not os.path.exists(pdb_path):
    urllib.request.urlretrieve(f'https://files.rcsb.org/download/{pdb_id}.pdb', pdb_path)

# Extract protein + ligand
u = mda.Universe(pdb_path)
prot_sel = u.select_atoms(f'protein and chainID {TARGET_CHAIN}')
lig_sel = u.select_atoms(f'not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4 and chainID {TARGET_CHAIN}')
if len(lig_sel) < 3:
    lig_sel = u.select_atoms('not protein and not resname HOH WAT SOL GOL PEG EDO PG4 DMS ACT BME CL NA MG ZN CA SO4')

clean_pdb = os.path.join(WORK_DIR, f'{pdb_id}_clean.pdb')
lig_mol2 = os.path.join(WORK_DIR, f'{pdb_id}_lig.mol2')
prot_sel.write(clean_pdb)

lig_tmp = os.path.join(WORK_DIR, f'{pdb_id}_lig_tmp.pdb')
lig_sel.write(lig_tmp)
mol = list(pybel.readfile(format='pdb', filename=lig_tmp))[0]
out = pybel.Outputfile(filename=lig_mol2, format='mol2', overwrite=True)
out.write(mol); out.close()
os.remove(lig_tmp)

# PDBFixer
prot_H = os.path.join(WORK_DIR, f'{pdb_id}_clean_H.pdb')
fixer = PDBFixer(filename=clean_pdb)
fixer.findMissingResidues(); fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues(); fixer.removeHeterogens(True)
fixer.findMissingAtoms(); fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)
with open(prot_H, 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

# Receptor PDBQT
rec_qt = os.path.join(WORK_DIR, f'{pdb_id}_rec.pdbqt')
mol = list(pybel.readfile(format='pdb', filename=prot_H))[0]
out = pybel.Outputfile(filename=rec_qt+'.tmp', format='pdbqt', overwrite=True)
out.write(mol); out.close()
skip_tags = ('ROOT','ENDROOT','BRANCH','ENDBRANCH','TORSDOF')
skip_kw = ('torsion','active','between atoms','status')
with open(rec_qt+'.tmp') as f: raw = f.readlines()
with open(rec_qt, 'w') as f:
    for l in raw:
        if l.startswith(skip_tags): continue
        if l.startswith('REMARK') and any(k in l.lower() for k in skip_kw): continue
        f.write(l)
os.remove(rec_qt+'.tmp')

print(f'Protein: {prot_H}')
print(f'Receptor PDBQT: {rec_qt}')
print(f'Co-crystal ligand: {lig_mol2} ({len(lig_sel)} atoms)')


## 3. Docking Box 설정 & 시각화


In [ ]:
#@title 3-1. Box 설정 {display-mode: "form"}
BOX_METHOD = "auto"  #@param ["auto", "residue", "manual"]
RESIDUE_LIST = "790, 858, 855"  #@param {type:"string"}
MANUAL_CENTER = "12.5, -3.0, 27.0"  #@param {type:"string"}
MANUAL_SIZE = "25.0, 25.0, 25.0"  #@param {type:"string"}
PADDING = 7.0  #@param {type:"number"}

def get_box_from_coords(coords, padding=7.0):
    minC, maxC = coords.min(axis=0), coords.max(axis=0)
    center = {'x': float((maxC[0]+minC[0])/2), 'y': float((maxC[1]+minC[1])/2), 'z': float((maxC[2]+minC[2])/2)}
    size = {'x': float(maxC[0]-minC[0]+2*padding), 'y': float(maxC[1]-minC[1]+2*padding), 'z': float(maxC[2]-minC[2]+2*padding)}
    return center, size

if BOX_METHOD == "auto":
    lig_u = mda.Universe(lig_mol2)
    center, size = get_box_from_coords(lig_u.atoms.positions, PADDING)
    print(f'Auto box from co-crystal ligand')
elif BOX_METHOD == "residue":
    res_nums = [int(r.strip()) for r in RESIDUE_LIST.split(',')]
    ref_prot = mda.Universe(prot_H)
    sel_str = ' or '.join([f'resid {r}' for r in res_nums])
    sel = ref_prot.select_atoms(sel_str)
    center, size = get_box_from_coords(sel.positions, PADDING)
    print(f'Residue box from {res_nums}')
elif BOX_METHOD == "manual":
    cx, cy, cz = [float(v) for v in MANUAL_CENTER.split(',')]
    sx, sy, sz = [float(v) for v in MANUAL_SIZE.split(',')]
    center = {'x': cx, 'y': cy, 'z': cz}
    size = {'x': sx, 'y': sy, 'z': sz}
    print('Manual box')

print(f'Center: ({center["x"]:.1f}, {center["y"]:.1f}, {center["z"]:.1f})')
print(f'Size:   ({size["x"]:.1f}, {size["y"]:.1f}, {size["z"]:.1f})')


In [ ]:
#@title 3-2. Box 3D 시각화 {display-mode: "form"}
view = py3Dmol.view(width=800, height=600)

with open(prot_H) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.6}})

with open(lig_mol2) as f:
    view.addModel(f.read(), 'mol2')
view.setStyle({'model': 1}, {'stick': {'color': 'green', 'radius': 0.2}})

# Docking box
view.addBox({
    'center': {'x': center['x'], 'y': center['y'], 'z': center['z']},
    'dimensions': {'w': size['x'], 'h': size['y'], 'd': size['z']},
    'color': 'blue', 'opacity': 0.15
})

# Box 외곽선 (Z축)
for dx in [-1, 1]:
    for dy in [-1, 1]:
        view.addCylinder({
            'start': {'x': center['x']+dx*size['x']/2, 'y': center['y']+dy*size['y']/2, 'z': center['z']-size['z']/2},
            'end': {'x': center['x']+dx*size['x']/2, 'y': center['y']+dy*size['y']/2, 'z': center['z']+size['z']/2},
            'radius': 0.1, 'color': 'blue', 'fromCap': False, 'toCap': False
        })

print('Blue box가 결합 부위를 감싸는지 확인하세요.')
print('너무 크거나 작으면 PADDING 또는 BOX_METHOD를 조절하세요.')
view.zoomTo()
view.show()


## 4. 전체 리간드 도킹


In [ ]:
#@title 4-1. 일괄 도킹 {display-mode: "form"}
EXHAUSTIVENESS = 32  #@param {type:"integer"}
N_POSES = 10         #@param {type:"integer"}
N_CPUS = 10          #@param {type:"integer"}

smina = os.path.join(BIN_DIR, 'smina')
results = []
t0 = time.time()

for i, lig in enumerate(LIGANDS):
    name = lig['name']
    smi = lig['smiles']
    print(f'[{i+1}/{len(LIGANDS)}] {name}...', end=' ', flush=True)

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        print('INVALID SMILES'); continue

    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)

    # Ligand PDBQT (meeko)
    lig_qt = os.path.join(WORK_DIR, f'{name}.pdbqt')
    for setup in MoleculePreparation().prepare(mol):
        s, ok, _ = PDBQTWriterLegacy.write_string(setup)
        if ok:
            with open(lig_qt, 'w') as f: f.write(s)
            break

    # Docking
    sdf_out = os.path.join(WORK_DIR, f'{name}_docked.sdf')
    subprocess.run([
        smina, '-r', rec_qt, '-l', lig_qt, '-o', sdf_out,
        '--center_x', str(center['x']), '--center_y', str(center['y']), '--center_z', str(center['z']),
        '--size_x', str(size['x']), '--size_y', str(size['y']), '--size_z', str(size['z']),
        '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(N_POSES),
        '--cpu', str(N_CPUS),
    ], capture_output=True)

    # Parse
    score = None
    if os.path.exists(sdf_out) and os.path.getsize(sdf_out) > 0:
        suppl = list(Chem.SDMolSupplier(sdf_out, removeHs=False))
        if suppl and suppl[0]:
            props = suppl[0].GetPropsAsDict()
            if 'minimizedAffinity' in props:
                try: score = float(props['minimizedAffinity'])
                except: pass

    ha = Chem.RemoveHs(mol).GetNumHeavyAtoms()
    mw = Descriptors.MolWt(Chem.RemoveHs(mol))
    results.append({
        'Name': name, 'SMILES': smi,
        'Score': round(score, 2) if score else None,
        'MW': round(mw, 1),
        'cLogP': round(Descriptors.MolLogP(Chem.MolFromSmiles(smi)), 2),
        'HBA': Descriptors.NumHAcceptors(Chem.MolFromSmiles(smi)),
        'HBD': Descriptors.NumHDonors(Chem.MolFromSmiles(smi)),
        'HA': ha,
        'LE': round(-score / ha, 3) if score and ha > 0 else None,
        'SDF': sdf_out,
    })
    print(f'{score:.2f} kcal/mol' if score else 'FAILED')

elapsed = time.time() - t0
print(f'\n=== {len([r for r in results if r["Score"]])} / {len(LIGANDS)} docked in {elapsed:.0f}s ===')


## 5. 결과 분석


In [ ]:
#@title 5-1. 스코어 랭킹 테이블 {display-mode: "form"}
df = pd.DataFrame(results).dropna(subset=['Score']).sort_values('Score')
display_cols = ['Name', 'Score', 'LE', 'MW', 'cLogP', 'HBA', 'HBD']
styled = df[display_cols].style \
    .background_gradient(subset=['Score'], cmap='RdYlGn') \
    .background_gradient(subset=['LE'], cmap='YlGn') \
    .format({'Score': '{:.2f}', 'LE': '{:.3f}', 'MW': '{:.1f}', 'cLogP': '{:.2f}'})
styled


In [ ]:
#@title 5-2. Score & LE 비교 차트 {display-mode: "form"}
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Score barplot
colors = ['#43A047' if s < -8 else '#FFB300' if s < -7 else '#E53935' for s in df['Score']]
axes[0].barh(df['Name'], df['Score'], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Docking Score (kcal/mol)')
axes[0].set_title('Docking Score Ranking')
axes[0].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5)
for _, row in df.iterrows():
    axes[0].text(row['Score']-0.1, row['Name'], f'{row["Score"]:.1f}', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

# Score vs MW scatter
axes[1].scatter(df['Score'], df['MW'], s=80, c=df['LE'], cmap='YlGn', edgecolors='black', linewidth=0.5)
for _, row in df.iterrows():
    axes[1].annotate(row['Name'], (row['Score'], row['MW']), fontsize=8, ha='center', va='bottom')
axes[1].set_xlabel('Score (kcal/mol)')
axes[1].set_ylabel('MW (Da)')
axes[1].set_title('Score vs MW')
axes[1].axhline(y=500, color='red', linestyle='--', alpha=0.3, label='Lipinski MW=500')
axes[1].legend()

# LE barplot
axes[2].barh(df['Name'], df['LE'].fillna(0), color='#1E88E5', edgecolor='black', linewidth=0.5)
axes[2].set_xlabel('Ligand Efficiency')
axes[2].set_title('Ligand Efficiency (LE)')
axes[2].axvline(x=0.3, color='green', linestyle='--', alpha=0.5, label='LE=0.3')
axes[2].legend()

plt.tight_layout()
plt.show()


In [ ]:
#@title 5-3. ProLIF 상호작용 분석 (Top 5) {display-mode: "form"}
top5 = df.head(5)
interaction_summary = []

prot_mol = Chem.MolFromPDBFile(prot_H, removeHs=False, sanitize=False)
Chem.SanitizeMol(prot_mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
prot_plf = plf.Molecule(prot_mol)

for _, row in top5.iterrows():
    sdf_out = row['SDF']
    if not os.path.exists(sdf_out): continue
    try:
        suppl = Chem.SDMolSupplier(sdf_out, sanitize=False, removeHs=False)
        ligs = []
        for mol in suppl:
            if mol is None: continue
            try:
                Chem.SanitizeMol(mol, SanitizeFlags.SANITIZE_ALL ^ SanitizeFlags.SANITIZE_PROPERTIES)
                ligs.append(plf.Molecule(mol))
            except: pass
        if ligs:
            fp = plf.Fingerprint()
            fp.run_from_iterable(ligs[:1], prot_plf, n_jobs=1)
            df_ifp = fp.to_dataframe()
            interactions = {}
            for col in df_ifp.columns:
                itype = str(col[2]) if isinstance(col, tuple) and len(col) > 2 else str(col)
                if df_ifp[col].any():
                    interactions[itype] = interactions.get(itype, 0) + 1
            interaction_summary.append({
                'Name': row['Name'], 'Score': row['Score'],
                'HBond': interactions.get('HBDonor', 0) + interactions.get('HBAcceptor', 0),
                'Hydrophobic': interactions.get('Hydrophobic', 0),
                'PiStack': interactions.get('PiStacking', 0),
                'Total': sum(interactions.values()),
            })
            print(f"{row['Name']}: {sum(interactions.values())} interactions")
    except Exception as e:
        print(f"{row['Name']}: ProLIF failed - {e}")

if interaction_summary:
    int_df = pd.DataFrame(interaction_summary)
    int_df


In [ ]:
#@title 5-4. Top 5 포즈 3D 비교 {display-mode: "form"}
view = py3Dmol.view(width=800, height=600)

# Protein
with open(prot_H) as f:
    view.addModel(f.read(), 'pdb')
view.setStyle({'model': 0}, {'cartoon': {'color': 'white', 'opacity': 0.5}})

# Docked poses
colors = ['#43A047', '#1E88E5', '#E53935', '#FF8F00', '#8E24AA']
model_idx = 1
legend_items = []

for i, (_, row) in enumerate(top5.iterrows()):
    sdf_out = row['SDF']
    if not os.path.exists(sdf_out): continue
    suppl = Chem.SDMolSupplier(sdf_out, removeHs=False, sanitize=False)
    if suppl and suppl[0]:
        block = Chem.MolToMolBlock(suppl[0])
        view.addModel(block, 'mol')
        view.setStyle({'model': model_idx}, {'stick': {'color': colors[i % len(colors)], 'radius': 0.2}})
        legend_items.append(f"{row['Name']}={colors[i % len(colors)]}")
        model_idx += 1

print(f'Top 5: {", ".join(legend_items)}')
view.zoomTo()
view.show()


## 6. 내보내기


In [ ]:
#@title 6-1. 결과 내보내기 {display-mode: "form"}
import shutil

# CSV
csv_path = os.path.join(WORK_DIR, 'screening_results.csv')
df.drop(columns=['SDF'], errors='ignore').to_csv(csv_path, index=False)
print(f'CSV: {csv_path}')

# Combined SDF
combined_sdf = os.path.join(WORK_DIR, 'screening_results.sdf')
writer = Chem.SDWriter(combined_sdf)
for _, row in df.iterrows():
    sdf_file = row.get('SDF', '')
    if sdf_file and os.path.exists(sdf_file):
        suppl = Chem.SDMolSupplier(sdf_file, removeHs=False, sanitize=False)
        if suppl and suppl[0]:
            mol = suppl[0]
            mol.SetProp('Name', str(row['Name']))
            mol.SetProp('Score', str(row['Score']))
            mol.SetProp('LE', str(row.get('LE', '')))
            writer.write(mol)
writer.close()
print(f'SDF: {combined_sdf}')

# PyMOL script
pml_path = os.path.join(WORK_DIR, 'screening_pymol.pml')
with open(pml_path, 'w') as f:
    f.write('reinitialize\nbg_color white\n\n')
    f.write(f'load {prot_H}, protein\nshow cartoon, protein\ncolor white, protein\n\n')
    pml_colors = ['green','cyan','yellow','salmon','orange']
    for i, (_, row) in enumerate(df.head(5).iterrows()):
        sdf = row['SDF']
        name = row['Name'].replace(' ', '_')
        c = pml_colors[i % len(pml_colors)]
        f.write(f'load {sdf}, {name}\nset all_states, 0\nframe 1\nshow sticks, {name}\ncolor {c}, {name}\n\n')
    f.write(f'select pocket, protein within 5 of {df.iloc[0]["Name"].replace(" ","_")}\nshow sticks, pocket\n')
    f.write(f'distance hb, {df.iloc[0]["Name"].replace(" ","_")}, pocket, mode=2\nset dash_color, yellow, hb\n')
    f.write(f'zoom {df.iloc[0]["Name"].replace(" ","_")}, 8\n')
print(f'PyMOL: {pml_path}')

# Zip
zip_path = os.path.join(os.path.dirname(WORK_DIR), f'{TARGET_PDB}_screening')
shutil.make_archive(zip_path, 'zip', WORK_DIR)
print(f'\nArchive: {zip_path}.zip')
try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Results at: {WORK_DIR}')
